## **Exercise 2: Using Transformer Models**

This exercise applies what you know of the architecture of transformer models by using pretrained models for inferencing. Specifically, this exercise uses the (**T**ext-**t**o-**T**ext **T**ransfer **T**ransformer) [T5 model.](https://huggingface.co/docs/transformers/en/model_doc/t5.)

T5 is an encoder-decoder transformer model. T5 transformers treat all NLP tasks as text-to-text transformations. In this framework, both the input and output for various NLP tasks are treated as textual sequences, making the model more adaptable and versatile.

Under this approach, T5 transformers can perform various tasks, including text classification, translation, summarization, question-answering, and more, by rephrasing the problem as a text generation task.


# Inferencing via Pipeline
Pipelines are objects that abstract most of the complex code from the Huggingface library, offering a simple API dedicated to several tasks, including Named Entity Recognition, Masked Language Modeling, Sentiment Analysis, Feature Extraction and Question Answering. This is the easiest way to use models that are ready for downstream tasks.

Learn more on usage and available tasks: https://huggingface.co/docs/transformers/v4.51.3/en/main_classes/pipelines#transformers.TextClassificationPipeline

In [1]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

In [2]:
import transformers
!pip install "transformers==4.46.3"
print("loaded version:", transformers.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

loaded version: 4.46.3


In [7]:
import torch
from transformers import pipeline

# This is an example pipeline for translation.

'''
The code uses an older version of the model. The model doesn't support text2text-generation anymore.
We used an older transformer model to run this pipeline instead.
'''

translation_pipeline = pipeline(
    task="text2text-generation",
    model="t5-large",
    torch_dtype=torch.float16,
    device=0
)
# Invoking the translation pipeline with a given input.
# Notice that the prefix "translate English to French:" that is related to the task is used before
# the sentence to be translated. This is how T5 or most encoder-decoder transformer models train
# sequence-to-sequence data. The prefixes (or language tags when used in translation), are ultimately
# learned by the model during training.
# T5 was trained on English-French, English-Romanian, and English-German translation.
translation_pipeline("translate English to French: I love Cebu", max_new_tokens=64)

[{'generated_text': "J'aime Cebu"}]

In [15]:
# YOU DO THIS!!!
# Fill in the right parameters for the sentiment analysis task.
# list of t5 tasks or prefixes: https://www.reddit.com/r/MachineLearning/comments/st97z9/d_where_can_i_find_a_list_of_t5_tasksprefixes/

# Summary from the Appendix D of T5 paper:
# Task   Prefix          Description
# SST-2	 sst2 sentence:	Performs sentiment analysis on a sentence.

sentiment_pipeline = pipeline(
    task="text2text-generation",
    model="t5-large",
    torch_dtype=torch.float16,
    device=0
)

print(sentiment_pipeline("sst2 sentence: I love Cebu", max_new_tokens=8))
# Sanity Check: output must be positive or negative

# After successfully running the script, generate sentiment classification for the following sentences:
# "Sulit sa bayad", "Okay na sa color green", "Sobrang iksi :(", "Ampanget!"
# What can you say about the results? Discuss.

sentences = ["Sulit sa bayad", "Okay na sa color green", "Sobrang iksi :(", "Ampanget!"]
for s in sentences:
    print(f"{s:25} -> {sentiment_pipeline('sst2 sentence: ' + s, max_new_tokens=8)[0]['generated_text']}")
    
# ANSWER:
# The positive sentiment for "I love Cebu" was correct because it was written in English and has positive words
# like "I love". However, the other sentences were written in our local language Tagalog (Filipino). This means
# their classification are not entirely accurate. Take "Ampanget!" for example, it has a positive sentiment even
# though it means ugly (negative sentiment) in Englishm, probably because of "!" which is usually used in positive 
# sentiments. "Sobrang iksi :(" was probably negative because of the sad emoticon at the end, which is also pretty 
# common in English vocabulary. "Okay na sa color green" was probably positive because of the workd "Okay". Overall, 
# the 4 last sentences are not accurately classified because the t5 was not trained for our local language Tagalog.

[{'generated_text': 'positive'}]
Sulit sa bayad            -> positive
Okay na sa color green    -> positive
Sobrang iksi :(           -> negative
Ampanget!                 -> positive


# Inferencing via model.generate()
This part of the exercise shows what happens under the pipeline's hood. generate() is used at inference time to generate texts. Following the encoder-decoder architecture, inference takes on the following tasks:

a. Tokenizing the input texts<br>
b. Generate output texts using the model<br>
c. Decoding the output texts<br>

In [16]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load pre-trained T5 model and tokenizer
# Each model has its own tokenizer to handle encoding of inputs and decoding of outputs to its readable form
model_name = "t5-large"  # or t5-base, t5-large
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [17]:
# Define the input text
input_text = "sst2 sentence: I hate this color"

# Tokenize input
input_ids = tokenizer(input_text, return_tensors="pt").input_ids
print(f'Input: {input_ids}')

# Generate output
outputs = model.generate(input_ids)#, max_length=2
print(f'Output: {outputs}')

# Decode output
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Decoded Output: {decoded_output }")

Input: tensor([[   3,    7,    7,   17,  357, 7142,   10,   27, 5591,   48,  945,    1]])
Output: tensor([[   0, 2841,    1]])
Decoded Output: negative


## AI Use Declaration
- I hereby declare upon my honor that I only used AI to understand the codes and concepts in this exercise.
- I did not use AI to generate any answers. All answers were written by me and are my own work.